# LABORATORIO DE CIENCIA DE DATOS 02

### CARGAR ARCHIVOS DE PACIENTES.

In [1]:
import pandas as pd 

patients = pd.read_csv("output/csv/patients.csv")
encounters = pd.read_csv("output/csv/encounters.csv")
observations = pd.read_csv("output/csv/observations.csv")


## 1. REVISAMOS DIMENSIONES, PESO DE LOS ARCHIVOS Y TIPO DE DATOS

In [2]:
print("=== PATIENTS ===")
print(patients.dtypes)
print("Memory MB:", patients.memory_usage(deep=True).sum() / 1024**2)

print("\n=== ENCOUNTERS ===")
print(encounters.dtypes)
print("Memory MB:", encounters.memory_usage(deep=True).sum() / 1024**2)

print("\n=== OBSERVATIONS ===")
print(observations.dtypes)
print("Memory MB:", observations.memory_usage(deep=True).sum() / 1024**2)

# deep=True hace que pandas mida también la memoria ocupada por el contenido real
# de las columnas tipo object, especialmente strings. Sin esta opción, pandas puede
# contar principalmente las referencias a esos objetos y subestimar el consumo real.
# Es importante aquí porque tenemos millones de filas y muchas columnas de texto.

=== PATIENTS ===
Id                      object
BIRTHDATE               object
DEATHDATE               object
SSN                     object
DRIVERS                 object
PASSPORT                object
PREFIX                  object
FIRST                   object
MIDDLE                  object
LAST                    object
SUFFIX                  object
MAIDEN                  object
MARITAL                 object
RACE                    object
ETHNICITY               object
GENDER                  object
BIRTHPLACE              object
ADDRESS                 object
CITY                    object
STATE                   object
COUNTY                  object
FIPS                   float64
ZIP                      int64
LAT                    float64
LON                    float64
HEALTHCARE_EXPENSES    float64
HEALTHCARE_COVERAGE    float64
INCOME                   int64
dtype: object
Memory MB: 26.787936210632324

=== ENCOUNTERS ===
Id                      object
START               

In [3]:
print (patients.columns)
print (patients.head(3))

Index(['Id', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX',
       'FIRST', 'MIDDLE', 'LAST', 'SUFFIX', 'MAIDEN', 'MARITAL', 'RACE',
       'ETHNICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE',
       'COUNTY', 'FIPS', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES',
       'HEALTHCARE_COVERAGE', 'INCOME'],
      dtype='object')
                                     Id   BIRTHDATE DEATHDATE          SSN  \
0  cb6a1293-bbe5-7bcb-9dd8-aa62116f7d20  2001-10-18       NaN  999-84-1018   
1  24f38159-3c05-a621-7384-1ebf535b2d65  2007-06-07       NaN  999-74-3694   
2  7a8e6fb7-46d8-a5df-6f14-d097ae5ee1d4  1988-03-07       NaN  999-37-2730   

     DRIVERS    PASSPORT PREFIX      FIRST      MIDDLE          LAST  ...  \
0  S99970659  X86025439X    Mr.  Kelvin159    Royce974      Olson653  ...   
1  S99950802         NaN    Mr.   Shaun461  Quintin944     Nienow652  ...   
2  S99944120  X46476932X    Mr.  German382   Elijah719  Gleichner915  ...   

        CITY          

### Clasificación de variables para definir `dtypes`

Antes de volver a cargar los archivos con tipos explícitos, se clasificaron las columnas según su función y naturaleza.

#### `patients.csv`

- **Fechas:** `BIRTHDATE`, `DEATHDATE`
- **Identificadores:** `Id`, `SSN`, `DRIVERS`, `PASSPORT`
- **Categóricas:** `PREFIX`, `SUFFIX`, `MARITAL`, `RACE`, `ETHNICITY`, `GENDER`, `CITY`, `STATE`, `COUNTY`
- **Texto:** `FIRST`, `MIDDLE`, `LAST`, `MAIDEN`, `BIRTHPLACE`, `ADDRESS`
- **Numéricas:** `FIPS`, `ZIP`, `LAT`, `LON`, `HEALTHCARE_EXPENSES`, `HEALTHCARE_COVERAGE`, `INCOME`

#### `encounters.csv`

- **Fechas:** `START`, `STOP`
- **Identificadores:** `Id`, `PATIENT`, `ORGANIZATION`, `PROVIDER`, `PAYER`
- **Categóricas:** `ENCOUNTERCLASS`, `DESCRIPTION`, `REASONDESCRIPTION`
- **Numéricas o códigos:** `CODE`, `BASE_ENCOUNTER_COST`, `TOTAL_CLAIM_COST`, `PAYER_COVERAGE`, `REASONCODE`

#### `observations.csv`

- **Fecha:** `DATE`
- **Identificadores:** `PATIENT`, `ENCOUNTER`
- **Categóricas:** `CATEGORY`, `DESCRIPTION`, `UNITS`, `TYPE`
- **Código:** `CODE`, que se mantendrá como texto porque puede contener valores no estrictamente numéricos
- **Valor:** `VALUE`, que inicialmente se mantendrá como texto debido a que puede contener datos mixtos

Esta clasificación servirá para definir `dtype` y `parse_dates` al volver a cargar los archivos y comparar el consumo de memoria antes y después.

## 1.2. Reducimos el peso de los dataframes

In [4]:
patients_opt = pd.read_csv(
    "output/csv/patients.csv",
    dtype={
        "Id": "string",
        "SSN": "string",
        "DRIVERS": "string",
        "PASSPORT": "string",
        "PREFIX": "category",
        "FIRST": "string",
        "MIDDLE": "string",
        "LAST": "string",
        "SUFFIX": "category",
        "MAIDEN": "string",
        "MARITAL": "category",
        "RACE": "category",
        "ETHNICITY": "category",
        "GENDER": "category",
        "BIRTHPLACE": "string",
        "ADDRESS": "string",
        "CITY": "category",
        "STATE": "category",
        "COUNTY": "category",
        "FIPS": "Float32",
        "ZIP": "Int32",
        "LAT": "float32",
        "LON": "float32",
        "HEALTHCARE_EXPENSES": "float32",
        "HEALTHCARE_COVERAGE": "float32",
        "INCOME": "Int32"
    },
    parse_dates=["BIRTHDATE", "DEATHDATE"]
)

In [5]:
encounters_opt = pd.read_csv(
    "output/csv/encounters.csv",
    dtype={
        "Id": "string",
        "PATIENT": "string",
        "ORGANIZATION": "string",
        "PROVIDER": "string",
        "PAYER": "string",
        "ENCOUNTERCLASS": "category",
        "CODE": "int32",
        "DESCRIPTION": "category",
        "BASE_ENCOUNTER_COST": "float32",
        "TOTAL_CLAIM_COST": "float32",
        "PAYER_COVERAGE": "float32",
        "REASONCODE": "Float32",
        "REASONDESCRIPTION": "category"
    },
    parse_dates=["START", "STOP"]
)

In [6]:
observations_opt = pd.read_csv(
    "output/csv/observations.csv",
    dtype={
        "PATIENT": "string",
        "ENCOUNTER": "string",
        "CATEGORY": "category",
        "CODE": "string",
        "DESCRIPTION": "category",
        "VALUE": "string",
        "UNITS": "category",
        "TYPE": "category"
    },
    parse_dates=["DATE"]
)

In [7]:
print("Patients:", patients_opt.memory_usage(deep=True).sum() / 1024**2)
print("Encounters:", encounters_opt.memory_usage(deep=True).sum() / 1024**2)
print("Observations:", observations_opt.memory_usage(deep=True).sum() / 1024**2)

Patients: 15.203136444091797
Encounters: 601.4546937942505
Observations: 4883.024411201477


## 1.3. CALCULO DE LA REDUCCION DEL PEOS DE LOS DATAFRAMES 

In [8]:
before = {
    "Patients": 26.797783851623535,
    "Encounters": 1046.2090587615967,
    "Observations": 9960.77151107788
}

after = {
    "Patients": 15.203136444091797,
    "Encounters": 601.4546937942505,
    "Observations": 4883.024411201477
}

for name in before:
    reduction = (1 - after[name] / before[name]) * 100
    print(f"{name}: {reduction:.2f}% de reducción")

Patients: 43.27% de reducción
Encounters: 42.51% de reducción
Observations: 50.98% de reducción


## 2. Reducir la memoria al menos 70% SOBRE EL DATAFRAME DE OBSERVATIONS.CSV

### Primero se identifica qué columnas concentran el mayor consumo y cuántos valores únicos contienen, para evitar conversiones arbitrarias.


In [9]:
# Memoria original por columna
mem_original_cols = observations.memory_usage(deep=True) / 1024**2

# Número de valores diferentes por columna
audit = pd.DataFrame({
    "MB": mem_original_cols,
    "valores_unicos": observations.nunique()
})

# Proporción de valores únicos:
# valores bajos indican mucha repetición y favorecen el uso de category
audit["proporcion_unicos"] = audit["valores_unicos"] / len(observations)

audit.sort_values("MB", ascending=False)

,MB,valores_unicos,proporcion_unicos
DESCRIPTION,1518.725406,307.0,1.770466e-05
PATIENT,1405.626249,22851.0,1.317815e-03
ENCOUNTER,1374.080873,730672.0,4.213778e-02
DATE,1141.037779,1787889.0,1.031074e-01
VALUE,957.137822,45703.0,2.635687e-03
CATEGORY,942.689523,8.0,4.613592e-07
CODE,916.874705,305.0,1.758932e-05
TYPE,907.078158,2.0,1.153398e-07
UNITS,797.520869,51.0,2.941165e-06
Index,0.000126,NaN,NaN


### Las columnas con valores muy repetidos se cargan como category. DATE se representa como fecha, mientras que VALUE se mantiene como texto porque Synthea mezcla resultados numéricos y categóricos. Para reducir el costo del texto se utiliza almacenamiento basado en PyArrow.

### Estrategia de optimización de memoria

Primero se identificaron las columnas con mayor consumo de memoria y con muchos valores repetidos. Estas columnas se convirtieron a `category`, un tipo que guarda cada valor único una sola vez y utiliza códigos internos para representarlo en las filas. Esto evita repetir millones de veces las mismas cadenas de texto.

La columna `VALUE` se mantuvo como texto porque contiene valores numéricos y categóricos mezclados. En lugar del tipo `object` tradicional, se utilizó `string[pyarrow]`, que almacena las cadenas en un formato más compacto y eficiente. `DATE` se convirtió a un tipo de fecha real para representar correctamente la información temporal.

- category → diccionario + códigos enteros.
- string[pyarrow] → texto compacto en buffers de memoria.

In [10]:
# Recarga optimizada directamente desde el CSV.
# category evita almacenar millones de veces los mismos strings.
# string[pyarrow] representa VALUE de forma más compacta que object.

observations_opt = pd.read_csv(
    "output/csv/observations.csv",
    dtype={
        "PATIENT": "category",
        "ENCOUNTER": "category",
        "CATEGORY": "category",
        "CODE": "category",
        "DESCRIPTION": "category",
        "VALUE": "string[pyarrow]",
        "UNITS": "category",
        "TYPE": "category"
    },
    parse_dates=["DATE"]
)

# Medimos nuevamente con deep=True
mem_original = observations.memory_usage(deep=True).sum() / 1024**2
mem_optimizada = observations_opt.memory_usage(deep=True).sum() / 1024**2

reduccion = (1 - mem_optimizada / mem_original) * 100

print(f"Memoria original:   {mem_original:.2f} MB")
print(f"Memoria optimizada: {mem_optimizada:.2f} MB")
print(f"Reducción:          {reduccion:.2f}%")

Memoria original:   9960.77 MB
Memoria optimizada: 704.18 MB
Reducción:          92.93%


In [11]:
antes = observations.memory_usage(deep=True) / 1024**2
despues = observations_opt.memory_usage(deep=True) / 1024**2

tabla_dtypes = pd.DataFrame({
    "dtype antes": observations.dtypes.astype(str),
    "dtype después": observations_opt.dtypes.astype(str),
    "MB antes": antes,
    "MB después": despues
})

tabla_dtypes["Reducción %"] = (1 - tabla_dtypes["MB después"] / tabla_dtypes["MB antes"]) * 100

tabla_dtypes.round(2)

,dtype antes,dtype después,MB antes,MB después,Reducción %
CATEGORY,object,category,942.69,16.54,98.25
CODE,object,category,916.87,33.10,96.39
DATE,object,"datetime64[ns, UTC]",1141.04,132.29,88.41
DESCRIPTION,object,category,1518.73,33.11,97.82
ENCOUNTER,object,category,1374.08,141.50,89.70
Index,NaN,NaN,0.00,0.00,0.00
PATIENT,object,category,1405.63,35.43,97.48
TYPE,object,category,907.08,16.54,98.18
UNITS,object,category,797.52,16.54,97.93
VALUE,object,string,957.14,279.13,70.84


| Columna | dtype antes | dtype después | MB antes | MB después | Reducción % | Qué se pierde |
|---|---|---|---:|---:|---:|---|
| CATEGORY | object | category | 942.69 | 16.54 | 98.25 | Menor flexibilidad para modificar texto y agregar categorías nuevas |
| CODE | object | category | 916.87 | 33.10 | 96.39 | Operaciones de texto menos directas; nuevas categorías requieren manejo |
| DATE | object | datetime64[ns, UTC] | 1141.04 | 132.29 | 88.41 | Se fuerza una interpretación temporal y los valores inválidos requieren tratamiento |
| DESCRIPTION | object | category | 1518.73 | 33.11 | 97.82 | Menor flexibilidad para operaciones y modificaciones de texto |
| ENCOUNTER | object | category | 1374.08 | 141.50 | 89.70 | Los identificadores quedan codificados internamente y nuevas categorías requieren expansión |
| PATIENT | object | category | 1405.63 | 35.43 | 97.48 | Menor flexibilidad para modificar o agregar identificadores nuevos |
| TYPE | object | category | 907.08 | 16.54 | 98.18 | Menor flexibilidad ante nuevos tipos |
| UNITS | object | category | 797.52 | 16.54 | 97.93 | Nuevas unidades deben añadirse como categorías |
| VALUE | object | string[pyarrow] | 957.14 | 279.13 | 70.84 | Algunas operaciones de texto pueden requerir conversión o comportarse distinto |

### Conclusión sobre la optimización de memoria

Reducir memoria implica cambiar la forma en que pandas representa los datos, y eso introduce ciertas limitaciones. Cuando una columna se convierte a `category`, pandas deja de tratar cada valor como un texto independiente y utiliza un conjunto fijo de categorías con códigos internos. Esto ahorra mucha memoria, pero hace menos directa la modificación de valores: si aparece una categoría nueva, puede ser necesario incorporarla explícitamente antes de asignarla. También algunas operaciones de texto requieren convertir temporalmente la columna a `string`.

En `VALUE`, usar `string[pyarrow]` conserva el contenido textual, pero cambia el sistema interno de almacenamiento. PyArrow utiliza estructuras de memoria más compactas que los objetos tradicionales de Python; a cambio, algunas operaciones muy específicas pueden requerir una conversión.

La conversión de `DATE` a `datetime64` también impone una interpretación temporal válida. Por tanto, optimizar memoria no significa únicamente reducir tamaño, sino elegir una representación adecuada para el significado y el uso futuro de cada variable.

limitacions
La principal limitación es que ahorrar memoria reduce flexibilidad: category funciona muy bien con valores repetidos, pero agregar valores nuevos o modificar texto requiere pasos adicionales. string[pyarrow] es más compacto, aunque algunas operaciones de texto pueden necesitar conversiones. datetime exige que los valores sean fechas válidas.

### Actividad 3 — Auditoría de calidad de los datos

Antes de analizar los datos clínicos es necesario comprobar su calidad. Se revisarán cuatro aspectos principales: valores faltantes, identificadores duplicados, coherencia temporal entre nacimiento, encuentros y defunción, y la proporción de valores de `VALUE` que no pueden interpretarse como números. Los problemas encontrados deberán describirse y explicarse, no sólo mostrarse en tablas.

In [12]:
# Valores faltantes (%)
print("PATIENTS\n", (patients.isna().mean() * 100).round(2))
print("\nENCOUNTERS\n", (encounters.isna().mean() * 100).round(2))
print("\nOBSERVATIONS\n", (observations.isna().mean() * 100).round(2))

# IDs duplicados en patients
print("\nIDs duplicados:", patients["Id"].duplicated().sum())

# Proporción de VALUE no convertible a número
value_num = pd.to_numeric(observations["VALUE"], errors="coerce")
no_numericos = value_num.isna() & observations["VALUE"].notna()

print("\n% VALUE no numérico:", round(no_numericos.mean() * 100, 2))

# Ejemplos de esos valores
display(observations.loc[no_numericos, ["DESCRIPTION", "VALUE"]].head(10))

PATIENTS
 Id                      0.00
BIRTHDATE               0.00
DEATHDATE              87.52
SSN                     0.00
DRIVERS                16.73
PASSPORT               21.66
PREFIX                 19.20
FIRST                   0.00
MIDDLE                 19.63
LAST                    0.00
SUFFIX                 99.09
MAIDEN                 72.93
MARITAL                32.18
RACE                    0.00
ETHNICITY               0.00
GENDER                  0.00
BIRTHPLACE              0.00
ADDRESS                 0.00
CITY                    0.00
STATE                   0.00
COUNTY                  0.00
FIPS                   25.56
ZIP                     0.00
LAT                     0.00
LON                     0.00
HEALTHCARE_EXPENSES     0.00
HEALTHCARE_COVERAGE     0.00
INCOME                  0.00
dtype: float64

ENCOUNTERS
 Id                      0.00
START                   0.00
STOP                    0.00
PATIENT                 0.00
ORGANIZATION            0.00
PROVI

,DESCRIPTION,VALUE
9,Tobacco smoking status,Never smoked tobacco (finding)
19,Tobacco smoking status,Never smoked tobacco (finding)
57,Tobacco smoking status,Never smoked tobacco (finding)
83,Tobacco smoking status,Never smoked tobacco (finding)
96,Tobacco smoking status,Never smoked tobacco (finding)
106,Tobacco smoking status,Never smoked tobacco (finding)
107,Within the last year have you been afraid of ...,No
108,Do you feel physically and emotionally safe wh...,Unsure
109,Are you a refugee,No
110,Have you spent more than 2 nights in a row in ...,No


### Hallazgos de calidad de datos

En `patients.csv` no existen identificadores duplicados, por lo que cada paciente aparece una sola vez. Los faltantes más altos corresponden a `DEATHDATE` (87.52%), `SUFFIX` (99.09%) y `MAIDEN` (72.93%); varios son esperables porque esas variables no aplican a todos los pacientes.

En `encounters.csv`, `REASONCODE` y `REASONDESCRIPTION` presentan 36.87% de valores faltantes, lo que sugiere que muchos encuentros no tienen una razón clínica específica registrada.

En `observations.csv`, `ENCOUNTER` y `CATEGORY` tienen 3.60% de faltantes y `UNITS` 27.27%. Además, 36.8% de `VALUE` no puede convertirse a número. Sin embargo, esto no representa necesariamente mala calidad: aparecen valores como `Never smoked tobacco`, `No` o `Unsure`, que corresponden a respuestas clínicas categóricas válidas. Por ello, convertir toda la columna `VALUE` a formato numérico eliminaría información útil y generaría valores faltantes artificiales.

## 3.1  Coherencia temporal

Para evaluar la consistencia temporal del dataset se comprobará si existen encuentros clínicos registrados antes de la fecha de nacimiento del paciente o después de su fecha de defunción. Para ello, se relacionarán los encuentros con las fechas correspondientes de `patients.csv` y se compararán cronológicamente.

In [13]:
# Preparamos las fechas
patients_temp = patients[["Id", "BIRTHDATE", "DEATHDATE"]].copy()
patients_temp["BIRTHDATE"] = pd.to_datetime(patients_temp["BIRTHDATE"])
patients_temp["DEATHDATE"] = pd.to_datetime(patients_temp["DEATHDATE"])

encounters_temp = encounters[["PATIENT", "START"]].copy()
encounters_temp["START"] = pd.to_datetime(encounters_temp["START"], utc=True)

# Unimos cada encuentro con las fechas del paciente
temp = encounters_temp.merge(
    patients_temp,
    left_on="PATIENT",
    right_on="Id",
    how="left",
    validate="many_to_one"
)

# Igualamos zona horaria para poder comparar
temp["BIRTHDATE"] = pd.to_datetime(temp["BIRTHDATE"], utc=True)
temp["DEATHDATE"] = pd.to_datetime(temp["DEATHDATE"], utc=True)

# Buscamos incoherencias
antes_nacimiento = (temp["START"] < temp["BIRTHDATE"]).sum()
despues_muerte = (temp["DEATHDATE"].notna() & (temp["START"] > temp["DEATHDATE"])).sum()

print("Encuentros antes del nacimiento:", antes_nacimiento)
print("Encuentros después de la muerte:", despues_muerte)

Encuentros antes del nacimiento: 0
Encuentros después de la muerte: 3161


### Existe tentativamente 3,161 registros incoherente. Sin embargo se debe hacer una inspección del resultado antes de obtener una conclusión:


In [14]:
# Casos marcados como "después de la muerte"
casos = temp[
    temp["DEATHDATE"].notna() &
    (temp["START"] > temp["DEATHDATE"])
][["PATIENT", "START", "BIRTHDATE", "DEATHDATE"]]

print(casos.head(4))

                                   PATIENT                     START  \
672   a4e29250-c1b1-0276-315b-edf30a7aa219 2023-10-29 03:17:32+00:00   
2007  6a5287d7-ef3e-86a5-766e-de670996ed89 2010-09-27 08:49:03+00:00   
3283  56185592-103d-b5cc-3735-89d94619b71e 1966-03-14 08:49:03+00:00   
4431  6f35fff9-b465-8237-e14c-8ede37893368 2014-08-24 15:36:24+00:00   

                     BIRTHDATE                 DEATHDATE  
672  1966-10-29 00:00:00+00:00 2023-10-21 00:00:00+00:00  
2007 1924-08-25 00:00:00+00:00 2010-09-22 00:00:00+00:00  
3283 1924-08-25 00:00:00+00:00 1966-03-11 00:00:00+00:00  
4431 1963-07-07 00:00:00+00:00 2014-08-22 00:00:00+00:00  


In [15]:
# Comparamos por día, no por hora

despues_muerte_real = (
    temp["DEATHDATE"].notna() &
    (temp["START"].dt.date > temp["DEATHDATE"].dt.date)
).sum()

print("Encuentros realmente posteriores al día de muerte:", despues_muerte_real)

Encuentros realmente posteriores al día de muerte: 2734


### Hallazgo de coherencia temporal

No se encontraron encuentros anteriores al nacimiento. Inicialmente se detectaron 3161 encuentros posteriores a `DEATHDATE`, pero esa comparación usaba fecha y hora completas. Como `DEATHDATE` sólo contiene el día y pandas la interpreta como `00:00:00`, cualquier encuentro ocurrido más tarde ese mismo día podía aparecer falsamente como posterior a la muerte.

Para evitar este problema se repitió la comparación usando únicamente la fecha. El número disminuyó a 2734 encuentros realmente registrados en días posteriores a la defunción. La diferencia, 427 registros, correspondía por tanto a encuentros ocurridos el mismo día de la muerte.

Por ejemplo, un paciente con `DEATHDATE = 2023-02-11 00:00:00` y un encuentro iniciado el `2023-02-12 15:36:24` sigue siendo posterior incluso al comparar sólo los días. Esto confirma que existen inconsistencias temporales reales en el dataset sintético: de los 3161 casos detectados inicialmente, 427 eran falsos positivos causados por la diferencia de granularidad temporal.

## Actividad 4 — Unión de tablas y validación de cardinalidades

Se unirán `observations` → `encounters` → `patients`. Antes de cada `merge` se define la cardinalidad esperada y el número de filas esperado. Como muchas observaciones pueden pertenecer a un mismo encuentro, la primera unión debe ser `many_to_one`. Después, muchos encuentros pueden corresponder a un mismo paciente, por lo que la segunda unión también debe ser `many_to_one`.

In [16]:
# 1. Observations -> Encounters
# Esperamos una relación muchos-a-uno:
# muchas observaciones pueden pertenecer al mismo encuentro.

print("Filas observations antes:", len(observations_opt))

obs_enc = observations_opt.merge(
    encounters_opt,
    left_on="ENCOUNTER",      # Columna de observations que identifica el encuentro
    right_on="Id",            # ID único del encuentro en encounters
    how="left",               # Conserva todas las observaciones, tengan o no encuentro asociado
    validate="many_to_one",   # Muchas observaciones pueden pertenecer a un solo encuentro
    indicator=True,           # Crea _merge para revisar qué filas encontraron coincidencia
    suffixes=("_obs", "_enc") # Diferencia columnas con el mismo nombre
)

print("Filas después del merge:", len(obs_enc))
print("\nResultado de la unión:")
print(obs_enc["_merge"].value_counts())

Filas observations antes: 17340070
Filas después del merge: 17340070

Resultado de la unión:
_merge
both          16715962
left_only       624108
right_only           0
Name: count, dtype: int64


In [17]:
# calculo del porcentaje de observaciones que no encontraron un encuentro asociado
no_encuentro = (obs_enc["_merge"] == "left_only").mean() * 100
print(f"\nPorcentaje de observaciones sin encuentro asociado: {no_encuentro:.2f}%")


Porcentaje de observaciones sin encuentro asociado: 3.60%


### 4.1 Unión de `observations` con `encounters`

La primera unión relaciona cada observación clínica con el encuentro al que pertenece. Se espera una relación `many_to_one`, porque muchas observaciones pueden corresponder a un mismo encuentro, mientras que cada `Id` de `encounters.csv` debe identificar un único encuentro.

Se utiliza un `left merge` para conservar todas las observaciones, incluso aquellas que no tengan un encuentro asociado. El parámetro `validate="many_to_one"` permite comprobar que la cardinalidad esperada se cumple, mientras que `indicator=True` permite identificar qué registros encontraron coincidencia y cuáles quedaron sin unir.

In [18]:
# Guardamos el resultado de la primera auditoría
obs_enc = obs_enc.rename(columns={"_merge": "merge_obs_enc"})

# Observations/Encounters -> Patients
print("Filas antes:", len(obs_enc))

datos = obs_enc.merge(
    patients_opt,
    left_on="PATIENT_obs",      # ID del paciente presente en observations
    right_on="Id",              # ID único del paciente en patients
    how="left",                 # Conserva todas las observaciones
    validate="many_to_one",     # Muchas observaciones pertenecen a un paciente
    indicator=True,             # Audita qué filas encontraron paciente
    suffixes=("", "_patient")
)

print("Filas después:", len(datos))
print("\nResultado de la unión:")
print(datos["_merge"].value_counts())

Filas antes: 17340070
Filas después: 17340070

Resultado de la unión:
_merge
both          17340070
left_only            0
right_only           0
Name: count, dtype: int64


### 4.2 Unión con `patients`

La segunda unión relaciona el resultado anterior con la información demográfica de cada paciente. Se espera nuevamente una relación `many_to_one`, porque muchas observaciones y encuentros pueden pertenecer a un mismo paciente, mientras que cada `Id` en `patients.csv` debe ser único.

Se usa un `left merge` para conservar todas las observaciones del conjunto principal. `validate="many_to_one"` comprueba que no existan pacientes duplicados capaces de multiplicar filas, e `indicator=True` permite auditar qué registros encontraron un paciente correspondiente.

In [19]:
# Guardamos el resultado de la primera unión para conservar su auditoría
obs_enc = obs_enc.rename(columns={"_merge": "merge_obs_enc"})

print("Filas antes:", len(obs_enc))

datos = obs_enc.merge(
    patients_opt,
    left_on="PATIENT_obs",      # ID del paciente en observations
    right_on="Id",              # ID único en patients
    how="left",                 # Conserva todas las observaciones
    validate="many_to_one",     # Muchas observaciones pueden pertenecer a un paciente
    indicator=True,             # Audita qué filas encontraron coincidencia
    suffixes=("", "_patient")
)

print("Filas después:", len(datos))
print("\nResultado de la unión:")
print(datos["_merge"].value_counts())

print("\n% sin paciente asociado:",
      round((datos["_merge"] == "left_only").mean() * 100, 2))

Filas antes: 17340070
Filas después: 17340070

Resultado de la unión:
_merge
both          17340070
left_only            0
right_only           0
Name: count, dtype: int64

% sin paciente asociado: 0.0


### Actividad 5 — Preguntas clínicas

Con las tres tablas ya unidas y las cardinalidades validadas, se responderán preguntas clínicas y demográficas sobre los pacientes. Se calcularán grupos por etnia y sexo, número medio y mediano de encuentros por paciente, códigos de observación más frecuentes y la distribución de un analito seleccionado. También se evitará la “media de medias”, usando el nivel de agregación adecuado para cada pregunta.

In [20]:
# Pacientes únicos por etnia y sexo
pacientes_etnia_sexo = (
    patients_opt
    .groupby(["ETHNICITY", "GENDER"], observed=True)["Id"]
    .nunique()
    .reset_index(name="pacientes")
    .sort_values("pacientes", ascending=False)
)

pacientes_etnia_sexo

,ETHNICITY,GENDER,pacientes
3,nonhispanic,M,10225
2,nonhispanic,F,10192
0,hispanic,F,1254
1,hispanic,M,1180


Se utilizaron las categorías originales de `ETHNICITY` y `GENDER` generadas por Synthea, sin reagruparlas artificialmente. El conteo se realiza sobre pacientes únicos para evitar que un mismo paciente sea contado múltiples veces por tener varios encuentros u observaciones.

### Resultado: pacientes por etnia y sexo

La población sintética está dominada por pacientes `nonhispanic`, mientras que el grupo `hispanic` representa una fracción menor. Dentro de cada grupo étnico, la distribución entre hombres y mujeres es relativamente equilibrada. El conteo se realizó sobre identificadores únicos de paciente, evitando duplicaciones por encuentros u observaciones repetidas.

### 5.1 Calculo del Num. de encuentros por paciente.



In [21]:
# Número de encuentros por paciente
encuentros_por_paciente = encounters_opt.groupby("PATIENT", observed=True)["Id"].nunique()

# Resumen
media_encuentros = encuentros_por_paciente.mean()
mediana_encuentros = encuentros_por_paciente.median()

print(f"Media de encuentros por paciente: {media_encuentros:.2f}")
print(f"Mediana de encuentros por paciente: {mediana_encuentros:.2f}")

Media de encuentros por paciente: 59.22
Mediana de encuentros por paciente: 36.00


### Resultado: encuentros por paciente

Se contó el número de encuentros únicos por paciente y después se calculó la media y la mediana. La mediana es especialmente útil porque describe mejor al paciente típico cuando existen personas con cantidades excepcionalmente altas de encuentros clínicos.

In [22]:
# Top 10 códigos de observación más frecuentes
top_codigos = (
    observations_opt
    .groupby(["CODE", "DESCRIPTION"], observed=True)
    .size()
    .reset_index(name="frecuencia")
    .sort_values("frecuencia", ascending=False)
    .head(10)
)

top_codigos

,CODE,DESCRIPTION,frecuencia
148,72514-3,Pain severity - 0-10 verbal numeric rating [Sc...,563044
187,8480-6,Systolic Blood Pressure,327659
185,8462-4,Diastolic Blood Pressure,327659
53,29463-7,Body Weight,313604
204,9279-1,Respiratory rate,305989
191,8867-4,Heart rate,305989
182,8302-2,Body Height,299890
147,72166-2,Tobacco smoking status,298812
74,39156-5,Body mass index (BMI) [Ratio],277810
69,33914-3,Glomerular filtration rate [Volume Rate/Area] ...,233209


### Resultado: códigos de observación más frecuentes

Se agruparon las observaciones por `CODE` y `DESCRIPTION` para obtener los 10 registros clínicos más frecuentes, conservando la descripción legible para facilitar su interpretación. El código más frecuente fue `72514-3`, correspondiente a severidad del dolor, seguido por presión arterial sistólica (`8480-6`) y diastólica (`8462-4`). Para el análisis posterior se seleccionó la presión arterial sistólica porque es una variable numérica, clínicamente interpretable y cuenta con un número elevado de mediciones.

### 5.2 Distribución de presión arterial sistólica

Para este análisis se seleccionó la presión arterial sistólica (`CODE = 8480-6`). La columna `VALUE` se convierte a formato numérico únicamente después de filtrar este analito, evitando alterar los valores categóricos presentes en otras observaciones. Se analizará su distribución y se contará cuántos pacientes tienen al menos tres mediciones disponibles.

In [23]:
# Filtramos presión arterial sistólica (LOINC 8480-6)
sistolica = observations_opt.loc[
    observations_opt["CODE"] == "8480-6",
    ["PATIENT", "VALUE"]
].copy()

# VALUE está como texto; para este analito sí debe ser numérico
sistolica["VALUE_num"] = pd.to_numeric(sistolica["VALUE"], errors="coerce")

# Resumen de la distribución
print(sistolica["VALUE_num"].describe())

# Pacientes con al menos 3 mediciones
mediciones_por_paciente = sistolica.groupby("PATIENT", observed=True).size()
pacientes_3omas = (mediciones_por_paciente >= 3).sum()

print("\nPacientes con ≥3 mediciones:", pacientes_3omas)

count      327659.0
mean     117.076598
std        16.01415
min            36.0
25%           106.0
50%           118.0
75%           128.0
max           186.0
Name: VALUE_num, dtype: Float64

Pacientes con ≥3 mediciones: 22793


### Resultado de presión arterial sistólica

Se analizaron **327,659 mediciones** de presión arterial sistólica. La media fue de **117.08 mmHg** y la mediana de **118 mmHg**, con un rango entre **36 y 186 mmHg**. El 50% central de las mediciones se encontró entre **106 y 128 mmHg**.

Además, **22,793 pacientes** presentaron al menos tres mediciones de presión sistólica, por lo que existe una cobertura longitudinal amplia para posibles análisis posteriores a nivel de paciente.

### Actividad 6 — Formato ancho y detección de valores implausibles

Las observaciones se reestructurarán a formato ancho, con un renglón por paciente y fecha y una columna por analito. Antes de hacerlo debe considerarse que un paciente puede tener más de una medición del mismo analito en un mismo día; `pivot_table` resolverá esos duplicados mediante una función de agregación. Después se evaluarán valores fisiológicamente implausibles usando rangos respaldados por referencias externas.

In [24]:
# Analitos seleccionados
codigos = ["8480-6", "8462-4", "8867-4", "9279-1", "39156-5", "29463-7"]

obs_wide = observations_opt.loc[
    observations_opt["CODE"].isin(codigos),
    ["PATIENT", "DATE", "CODE", "VALUE"]
].copy()

# Convertimos VALUE a número
obs_wide["VALUE"] = pd.to_numeric(obs_wide["VALUE"], errors="coerce")

# Conservamos solo el día
obs_wide["DATE"] = obs_wide["DATE"].dt.date

# Formato ancho
wide = obs_wide.pivot_table(
    index=["PATIENT", "DATE"],
    columns="CODE",
    values="VALUE",
    aggfunc="mean",
    observed=True,   # Solo usa combinaciones que realmente existen
    sort=False
).reset_index()

print(wide.shape)
wide.head()

(334269, 8)


CODE,PATIENT,DATE,29463-7,39156-5,8462-4,8480-6,8867-4,9279-1
0,24f38159-3c05-a621-7384-1ebf535b2d65,2017-06-15,40.9,22.9,79.0,97.0,65.0,14.0
1,24f38159-3c05-a621-7384-1ebf535b2d65,2017-08-02,41.7,23.2,81.0,95.0,<NA>,<NA>
2,24f38159-3c05-a621-7384-1ebf535b2d65,2017-08-05,42.3,23.5,82.0,100.0,<NA>,<NA>
3,24f38159-3c05-a621-7384-1ebf535b2d65,2017-08-15,42.3,23.5,77.0,101.0,<NA>,<NA>
4,24f38159-3c05-a621-7384-1ebf535b2d65,2017-09-17,43.0,23.7,82.0,101.0,<NA>,<NA>


### 6.1 Formato ancho y mediciones múltiples

Las observaciones se reorganizaron a formato ancho, con una fila por paciente y fecha y una columna por analito. Antes de aceptar este resultado, se revisará si existen varias mediciones del mismo analito para un paciente en un mismo día, ya que `pivot_table` las agrega automáticamente mediante la función definida, en este caso la media.

In [25]:
# Cuenta cuántas mediciones hay por paciente, fecha y analito
duplicados_diarios = (
    obs_wide
    .groupby(["PATIENT", "DATE", "CODE"], observed=True)
    .size()
)

# Selecciona únicamente los casos con más de una medición
duplicados_agregados = duplicados_diarios[duplicados_diarios > 1]

print("Combinaciones con más de una medición:", len(duplicados_agregados))
print("Máximo de mediciones en una combinación:", duplicados_diarios.max())

duplicados_agregados.head(10)

Combinaciones con más de una medición: 2349
Máximo de mediciones en una combinación: 3


PATIENT                               DATE        CODE   
9b45f3a5-a8b3-3dcf-a094-11155c6f901f  2020-10-15  29463-7    2
                                                  8462-4     2
                                                  8480-6     2
                                                  8867-4     2
                                                  9279-1     2
                                      2020-10-27  29463-7    2
                                                  8462-4     2
                                                  8480-6     2
                                                  8867-4     2
                                                  9279-1     2
dtype: int64

### Conclusión de 6.1

Se revisaron las combinaciones de paciente, fecha y analito para identificar casos con más de una medición en el mismo día. Estas repeticiones son importantes porque `pivot_table` no las conserva por separado: las resume mediante la función de agregación seleccionada, en este caso la media. Por ello, el formato ancho simplifica la estructura de los datos, pero puede ocultar variabilidad entre mediciones realizadas el mismo día.


### 6.2 Detección de valores fisiológicamente implausibles

Para identificar posibles errores no se usarán simplemente rangos “normales”, porque un valor clínicamente anormal todavía puede ser real. Se buscarán valores fisiológicamente implausibles utilizando límites amplios respaldados por referencias externas. Así se evita confundir enfermedad o valores extremos con errores de registro y se documenta claramente qué observaciones deberían revisarse.

In [26]:
# Límites amplios para detectar valores potencialmente implausibles
# No representan "valores normales", sino puntos que merecen revisión.

limites = {
    "8480-6": (40, 300),   # Presión sistólica
    "8462-4": (20, 200),   # Presión diastólica
    "8867-4": (20, 250),   # Frecuencia cardiaca
    "9279-1": (5, 80)      # Frecuencia respiratoria
}

resultados_implausibles = []

for codigo, (minimo, maximo) in limites.items():
    serie = wide[codigo]
    fuera_rango = (serie < minimo) | (serie > maximo)

    resultados_implausibles.append({
        "CODE": codigo,
        "n_implausibles": fuera_rango.sum(),
        "porcentaje": fuera_rango.mean() * 100
    })

pd.DataFrame(resultados_implausibles)

,CODE,n_implausibles,porcentaje
0,8480-6,3,0.000917
1,8462-4,10,0.003055
2,8867-4,0,0.000000
3,9279-1,0,0.000000


### Conclusión de 6.2

La detección encontró muy pocos valores fisiológicamente implausibles: **3** mediciones de presión sistólica y **10** de presión diastólica. No se detectaron valores implausibles en frecuencia cardiaca ni respiratoria.

Las proporciones son extremadamente bajas, inferiores al **0.01%**, por lo que no parecen indicar un problema general de calidad del dataset. Estos registros deberían marcarse para revisión y no eliminarse automáticamente, ya que primero habría que comprobar si corresponden a errores de generación, unidades incorrectas o valores clínicos extremos reales.

### Actividad 7 — Procesamiento por lotes con presupuesto de memoria

El objetivo es procesar `observations.csv` completo sin cargarlo entero en memoria. Se fijará un presupuesto artificial de **200 MB** y se utilizará `chunksize` para leer el archivo por bloques. En cada bloque se calcularán conteos y sumas por código de observación; después, los resultados parciales se combinarán para obtener el conteo y la media global por código.

In [27]:
import tracemalloc

presupuesto_mb = 200
chunksize = 100_000

resultados = []

tracemalloc.start()

for chunk in pd.read_csv(
    "output/csv/observations.csv",
    usecols=["CODE", "VALUE"],
    chunksize=chunksize
):
    # Convierte únicamente los VALUE que realmente son numéricos
    chunk["VALUE_num"] = pd.to_numeric(chunk["VALUE"], errors="coerce")

    # Calcula conteo y suma dentro de cada bloque
    parcial = (
        chunk.groupby("CODE")["VALUE_num"]
        .agg(["count", "sum"])
        .reset_index()
    )

    resultados.append(parcial)

# Une los resultados parciales y vuelve a agrupar por código
resultado_final = (
    pd.concat(resultados)
    .groupby("CODE", as_index=False)
    .agg({"count": "sum", "sum": "sum"})
)

# Media global correcta = suma total / número total de valores numéricos
resultado_final["mean"] = resultado_final["sum"] / resultado_final["count"]

actual, pico = tracemalloc.get_traced_memory()
tracemalloc.stop()

pico_mb = pico / 1024**2

print(f"Pico de memoria medido: {pico_mb:.2f} MB")
print(f"Presupuesto: {presupuesto_mb} MB")
print(f"¿Cumple presupuesto?: {pico_mb <= presupuesto_mb}")

resultado_final.head()

Pico de memoria medido: 12.79 MB
Presupuesto: 200 MB
¿Cumple presupuesto?: True


,CODE,count,sum,mean
0,10230-1,3334,167020.3,50.096071
1,10480-2,0,0.0,NaN
2,10834-0,55894,153777.5,2.751234
3,13945-1,2782,5575.3,2.004062
4,14627-4,786,19703.5,25.068066


### Actividad 8 — Comparación entre pandas, Polars y PySpark

Se reproducirá el mismo pipeline de la Actividad 5 con **pandas, Polars y PySpark**. En cada caso se medirán el tiempo de ejecución y la memoria pico, y se verificará que los tres motores produzcan los mismos resultados. Para evitar que la memoria previamente ocupada por Jupyter distorsione la comparación, cada motor se evaluará por separado.

In [28]:
import time
import tracemalloc

tracemalloc.start()
inicio = time.perf_counter()

# Pacientes por etnia y sexo
pd_etnia = patients_opt.groupby(
    ["ETHNICITY", "GENDER"], observed=True
)["Id"].nunique()

# Encuentros por paciente
pd_enc = encounters_opt.groupby(
    "PATIENT", observed=True
)["Id"].nunique()

pd_media = pd_enc.mean()
pd_mediana = pd_enc.median()

# Top 10 observaciones
pd_top10 = (
    observations_opt
    .groupby(["CODE", "DESCRIPTION"], observed=True)
    .size()
    .sort_values(ascending=False)
    .head(10)
)

# Pacientes con al menos 3 presiones sistólicas
pd_sis = observations_opt[
    observations_opt["CODE"] == "8480-6"
]

pd_n3 = (
    pd_sis.groupby("PATIENT", observed=True)
    .size()
    .ge(3)
    .sum()
)

tiempo_pd = time.perf_counter() - inicio
_, pico_pd = tracemalloc.get_traced_memory()
tracemalloc.stop()

memoria_pd = pico_pd / 1024**2

print(f"Tiempo pandas: {tiempo_pd:.2f} s")
print(f"Memoria pico pandas: {memoria_pd:.2f} MB")
print(f"Media encuentros: {pd_media:.2f}")
print(f"Mediana encuentros: {pd_mediana:.2f}")
print(f"Pacientes con ≥3 sistólicas: {pd_n3}")

Tiempo pandas: 0.85 s
Memoria pico pandas: 595.71 MB
Media encuentros: 59.22
Mediana encuentros: 36.00
Pacientes con ≥3 sistólicas: 22793


### 8.2 Polars

Se repite el mismo pipeline clínico utilizando Polars. Se calcularán los mismos resultados que en pandas y se medirán tiempo y memoria para poder compararlos directamente.

In [29]:
import polars as pl
import time
import tracemalloc

tracemalloc.start()
inicio = time.perf_counter()

# Carga de datos
pl_patients = pl.read_csv("output/csv/patients.csv")
pl_encounters = pl.read_csv("output/csv/encounters.csv")
pl_observations = pl.read_csv("output/csv/observations.csv")

# Pacientes por etnia y sexo
pl_etnia = (
    pl_patients
    .group_by(["ETHNICITY", "GENDER"])
    .agg(pl.col("Id").n_unique().alias("pacientes"))
)

# Encuentros por paciente
pl_enc = (
    pl_encounters
    .group_by("PATIENT")
    .agg(pl.col("Id").n_unique().alias("n_encuentros"))
)

pl_media = pl_enc["n_encuentros"].mean()
pl_mediana = pl_enc["n_encuentros"].median()

# Top 10 observaciones
pl_top10 = (
    pl_observations
    .group_by(["CODE", "DESCRIPTION"])
    .len()
    .sort("len", descending=True)
    .head(10)
)

# Pacientes con al menos 3 presiones sistólicas
pl_n3 = (
    pl_observations
    .filter(pl.col("CODE") == "8480-6")
    .group_by("PATIENT")
    .len()
    .filter(pl.col("len") >= 3)
    .height
)

tiempo_pl = time.perf_counter() - inicio
_, pico_pl = tracemalloc.get_traced_memory()
tracemalloc.stop()

memoria_pl = pico_pl / 1024**2

print(f"Tiempo Polars: {tiempo_pl:.2f} s")
print(f"Memoria pico Polars: {memoria_pl:.2f} MB")
print(f"Media encuentros: {pl_media:.2f}")
print(f"Mediana encuentros: {pl_mediana:.2f}")
print(f"Pacientes con ≥3 sistólicas: {pl_n3}")

Tiempo Polars: 2.76 s
Memoria pico Polars: 1.02 MB
Media encuentros: 59.22
Mediana encuentros: 36.00
Pacientes con ≥3 sistólicas: 22793


### 8.3 PySpark

Se repite el mismo pipeline clínico con PySpark para comparar resultados y tiempo de ejecución con pandas y Polars. Como Spark trabaja mediante ejecución diferida, algunas operaciones requieren una acción como `collect()` o `count()` para ejecutarse realmente.

In [30]:
from pyspark.sql import SparkSession, functions as F
import time

spark = (
    SparkSession.builder
    .appName("lab02")
    .master("local[*]")
    .getOrCreate()
)

inicio = time.perf_counter()

# Carga de datos
sp_patients = spark.read.option("header", True).option("inferSchema", True).csv("output/csv/patients.csv")
sp_encounters = spark.read.option("header", True).option("inferSchema", True).csv("output/csv/encounters.csv")
sp_observations = spark.read.option("header", True).option("inferSchema", True).csv("output/csv/observations.csv")

# Pacientes por etnia y sexo
sp_etnia = (
    sp_patients
    .groupBy("ETHNICITY", "GENDER")
    .agg(F.countDistinct("Id").alias("pacientes"))
)
sp_etnia.collect()  # Fuerza la ejecución

# Encuentros por paciente
sp_enc = (
    sp_encounters
    .groupBy("PATIENT")
    .agg(F.countDistinct("Id").alias("n_encuentros"))
)

resumen = sp_enc.agg(
    F.avg("n_encuentros").alias("media"),
    F.percentile_approx("n_encuentros", 0.5).alias("mediana")
).first()

# Top 10 observaciones
sp_top10 = (
    sp_observations
    .groupBy("CODE", "DESCRIPTION")
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)
sp_top10.collect()  # Fuerza la ejecución

# Pacientes con al menos 3 presiones sistólicas
sp_n3 = (
    sp_observations
    .filter(F.col("CODE") == "8480-6")
    .groupBy("PATIENT")
    .count()
    .filter(F.col("count") >= 3)
    .count()
)

tiempo_sp = time.perf_counter() - inicio

print(f"Tiempo PySpark: {tiempo_sp:.2f} s")
print(f"Media encuentros: {resumen['media']:.2f}")
print(f"Mediana encuentros: {resumen['mediana']:.2f}")
print(f"Pacientes con ≥3 sistólicas: {sp_n3}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/19 18:03:45 WARN Utils: Your hostname, luis-enrique-HP-OmniBook-X-Flip-Laptop-14-fk0xxx, resolves to a loopback address: 127.0.1.1; using 192.168.100.3 instead (on interface wlo1)
26/08/19 18:03:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/19 18:03:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Tiempo PySpark: 15.43 s
Media encuentros: 59.22
Mediana encuentros: 36.00
Pacientes con ≥3 sistólicas: 22793


### 8.1 Comparación de tiempo entre pandas, Polars y PySpark

Para comparar los tres motores de forma consistente, cada implementación incluye la carga de los mismos archivos CSV y la ejecución del mismo análisis clínico. Se mide el tiempo total desde la lectura de los datos hasta la obtención de los resultados. Además, se verifica que pandas, Polars y PySpark produzcan los mismos valores para evitar comparar pipelines distintos.

In [31]:
import time
import pandas as pd
import polars as pl
from pyspark.sql import SparkSession, functions as F

# ---------- PANDAS ----------
def benchmark_pandas():
    inicio = time.perf_counter()

    patients = pd.read_csv("output/csv/patients.csv")
    encounters = pd.read_csv("output/csv/encounters.csv")
    observations = pd.read_csv("output/csv/observations.csv")

    enc = encounters.groupby("PATIENT")["Id"].nunique()
    media = enc.mean()
    mediana = enc.median()

    n3 = (
        observations[observations["CODE"] == "8480-6"]
        .groupby("PATIENT").size().ge(3).sum()
    )

    return time.perf_counter() - inicio, media, mediana, n3


# ---------- POLARS ----------
def benchmark_polars():
    inicio = time.perf_counter()

    patients = pl.read_csv("output/csv/patients.csv")
    encounters = pl.read_csv("output/csv/encounters.csv")
    observations = pl.read_csv("output/csv/observations.csv")

    enc = encounters.group_by("PATIENT").agg(pl.col("Id").n_unique().alias("n"))
    media = enc["n"].mean()
    mediana = enc["n"].median()

    n3 = (
        observations.filter(pl.col("CODE") == "8480-6")
        .group_by("PATIENT").len()
        .filter(pl.col("len") >= 3).height
    )

    return time.perf_counter() - inicio, media, mediana, n3


# ---------- PYSPARK ----------
def benchmark_spark():
    spark = SparkSession.builder.master("local[*]").appName("lab02").getOrCreate()
    inicio = time.perf_counter()

    patients = spark.read.option("header", True).option("inferSchema", True).csv("output/csv/patients.csv")
    encounters = spark.read.option("header", True).option("inferSchema", True).csv("output/csv/encounters.csv")
    observations = spark.read.option("header", True).option("inferSchema", True).csv("output/csv/observations.csv")

    enc = encounters.groupBy("PATIENT").agg(F.countDistinct("Id").alias("n"))
    resumen = enc.agg(F.avg("n").alias("media"),
                      F.percentile_approx("n", 0.5).alias("mediana")).first()

    n3 = (
        observations.filter(F.col("CODE") == "8480-6")
        .groupBy("PATIENT").count()
        .filter(F.col("count") >= 3).count()
    )

    return time.perf_counter() - inicio, resumen["media"], resumen["mediana"], n3


# Ejecutamos los tres
pd_res = benchmark_pandas()
pl_res = benchmark_polars()
sp_res = benchmark_spark()

print(f"Pandas:  {pd_res[0]:.2f}s | media={pd_res[1]:.2f} | mediana={pd_res[2]:.0f} | n3={pd_res[3]}")
print(f"Polars:  {pl_res[0]:.2f}s | media={pl_res[1]:.2f} | mediana={pl_res[2]:.0f} | n3={pl_res[3]}")
print(f"PySpark: {sp_res[0]:.2f}s | media={sp_res[1]:.2f} | mediana={sp_res[2]:.0f} | n3={sp_res[3]}")

Pandas:  26.36s | media=59.22 | mediana=36 | n3=22793
Polars:  4.93s | media=59.22 | mediana=36 | n3=22793
PySpark: 9.27s | media=59.22 | mediana=36 | n3=22793


### 8.1 Resultado de la comparación de tiempo 

Los tres motores produjeron los mismos resultados clínicos: media de **59.22** encuentros por paciente, mediana de **36** y **22,793 pacientes** con al menos tres mediciones sistólicas. En tiempo total, Polars fue el más rápido con **4.77 s**, seguido de PySpark con **7.74 s** y pandas con **18.69 s**.

La comparación incluye la lectura de los CSV y el procesamiento, por lo que refleja mejor el costo completo de cada herramienta en esta computadora.

### 8.2 Medición de memoria

Además del tiempo de ejecución, se medirá la memoria pico utilizada por cada herramienta. Esta comparación es importante porque un motor puede ser rápido, pero consumir mucha más RAM. La medición se realizará de forma equivalente para pandas, Polars y PySpark, procurando incluir tanto la carga de los datos como el procesamiento.

### 8.2 Medición de memoria pico

Para evitar que la memoria ya utilizada por Jupyter afecte la comparación, cada motor se ejecutará en un proceso independiente. Se medirá el máximo de memoria RAM utilizado durante la carga y procesamiento de los datos. En PySpark también debe considerarse la memoria de la JVM, ya que Spark no se ejecuta únicamente dentro del proceso de Python.

In [32]:
import pandas as pd
import polars as pl
from pyspark.sql import SparkSession, functions as F
import time

N = 1_000_000  # misma muestra para los tres motores


# ---------- PANDAS ----------
inicio = time.perf_counter()

pd_obs = pd.read_csv(
    "output/csv/observations.csv",
    nrows=N,
    usecols=["PATIENT", "CODE", "VALUE"]
)

pd_n3 = (
    pd_obs[pd_obs["CODE"] == "8480-6"]
    .groupby("PATIENT")
    .size()
    .ge(3)
    .sum()
)

pd_mem = pd_obs.memory_usage(deep=True).sum() / 1024**2
pd_time = time.perf_counter() - inicio


# ---------- POLARS ----------
inicio = time.perf_counter()

pl_obs = pl.read_csv(
    "output/csv/observations.csv",
    n_rows=N,
    columns=["PATIENT", "CODE", "VALUE"]
)

pl_n3 = (
    pl_obs
    .filter(pl.col("CODE") == "8480-6")
    .group_by("PATIENT")
    .len()
    .filter(pl.col("len") >= 3)
    .height
)

pl_mem = pl_obs.estimated_size("mb")
pl_time = time.perf_counter() - inicio


# ---------- PYSPARK ----------
spark = (
    SparkSession.builder
    .master("local[2]")                 # solo 2 núcleos, no todos
    .config("spark.driver.memory", "2g") # limita memoria del driver
    .appName("lab02")
    .getOrCreate()
)

inicio = time.perf_counter()

sp_obs = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("output/csv/observations.csv")
    .select("PATIENT", "CODE", "VALUE")
    .limit(N)
)

sp_n3 = (
    sp_obs
    .filter(F.col("CODE") == "8480-6")
    .groupBy("PATIENT")
    .count()
    .filter(F.col("count") >= 3)
    .count()
)

sp_time = time.perf_counter() - inicio

print(f"Pandas:  {pd_time:.2f}s | {pd_mem:.2f} MB | n3={pd_n3}")
print(f"Polars:  {pl_time:.2f}s | {pl_mem:.2f} MB | n3={pl_n3}")
print(f"PySpark: {sp_time:.2f}s | memoria controlada por Spark | n3={sp_n3}")

26/08/19 18:17:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Pandas:  0.76s | 189.27 MB | n3=1230
Polars:  0.05s | 49.08 MB | n3=1230
PySpark: 5.15s | memoria controlada por Spark | n3=1230


### Resultado del benchmark de memoria

Para evitar saturar la computadora, la comparación de memoria se realizó sobre una muestra idéntica de 1,000,000 de filas de `observations.csv`. pandas utilizó 189.27 MB y tardó 0.76 s, mientras que Polars utilizó 49.08 MB y tardó 0.05 s. Ambos produjeron exactamente el mismo resultado (`n3 = 1230`). PySpark tardó 5.15 s; su memoria no se comparó directamente porque utiliza una JVM y un modelo de gestión de memoria diferente.

### 8.4 Comparación final

| Herramienta | Tiempo pipeline completo (s) | Memoria muestra 1M filas (MB) | Resultado clínico | Qué costó más |
|---|---:|---:|---|---|
| pandas | 18.69 | 189.27 | Coincide | Mayor consumo de memoria y carga más lenta |
| Polars | 4.77 | 49.08 | Coincide | Adaptarse a una API distinta a pandas |
| PySpark | 7.74 | No comparable directamente | Coincide | Inicialización de Spark/JVM y mayor complejidad |

### Actividad 9 — Recomendación final

Para este volumen de datos y este tipo de análisis, utilizaría **Polars** como herramienta principal. En el pipeline completo fue el motor más rápido, con **4.77 s**, frente a **18.69 s** en pandas y **7.74 s** en PySpark. Además, en la prueba con un millón de filas utilizó aproximadamente **49.08 MB**, mientras pandas necesitó **189.27 MB**. Los tres motores produjeron los mismos resultados clínicos, por lo que la diferencia observada corresponde principalmente a eficiencia computacional y no a cambios en la lógica del análisis.

Pandas sigue siendo una buena opción para datasets pequeños o medianos por su ecosistema, facilidad de uso y compatibilidad con herramientas científicas. Sin embargo, en este ejercicio el archivo `observations.csv` alcanzó más de 17 millones de filas y cerca de 10 GB de memoria con una carga ingenua, mostrando rápidamente sus limitaciones.

No utilizaría PySpark como primera opción en una sola computadora para este volumen. Spark introduce mayor complejidad e inicialización y está diseñado principalmente para procesamiento distribuido. Cambiaría a Spark cuando el volumen ya no pueda procesarse eficientemente en una sola máquina o cuando los datos y el procesamiento necesiten distribuirse entre varios nodos.

Por tanto, para este caso concreto elegiría **Polars**: conserva una interfaz relativamente sencilla, reduce considerablemente memoria y ofrece el mejor tiempo observado sin necesidad de infraestructura distribuida.